In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
import sys

sys.path.append(os.path.abspath('./src'))
import db_functions as dbf

import logging
from basic_logger import setup_logger
setup_logger()
db = dbf.DotaDB()

In [2]:
## Top hero winrates
columns=['id', 'didRadiantWin', 'gameVersionId', 'heroId', 'isRadiant', 'isVictory', 'displayName']
query = '''
    SELECT md.{0}, md.{1}, md.{2}, mp.{3}, mp.{4}, mp.{5}, hero_details.{6} 
    FROM match_details md
    INNER JOIN match_players mp
    ON md.id = mp.match_id
    INNER JOIN hero_details
    ON mp."heroId" = hero_details.id;
'''
df = db.query_select_to_df(query, table_name='match_details', columns=columns, identifiers=columns)
hero_stats = df.groupby(['displayName'])['isVictory'].aggregate(['mean', 'count'])
hero_stats.columns = ['Win rate', 'Games played']
hero_stats['Win rate'] = (hero_stats['Win rate']*100).round(2)
hero_stats = hero_stats.sort_values(by='Win rate', ascending=False)
hero_stats = hero_stats[hero_stats['Games played'] >= 100]
hero_stats.head(10)

,Win rate,Games played
displayName,,
Chen,58.87,462
Naga Siren,57.65,477
Bane,57.04,419
Ember Spirit,55.29,823
Riki,55.12,127
Faceless Void,55.09,265
Nature's Prophet,54.95,737
Venomancer,54.55,132
Clinkz,54.20,131


In [3]:
## Pick and ban rates
## IMPORTANT: matches were dropped that had no pick-ban phase and where a pick/ban was missing
query = '''
    SELECT mpb.*, hero_details.name, hero_details."displayName"
    FROM match_pick_bans mpb
    INNER JOIN hero_details
    ON mpb."heroId" = hero_details.id;
'''
results = db.query_select(query)
df = pd.DataFrame(results, columns=['id', 'matchId', 'isPick', 'heroId', 'order', 'isRadiant', 'heroName', 'displayName'])
orders = df.groupby('matchId')['order'].max().sort_values(ascending=True)
matches_to_drop = orders.loc[lambda x: x == 23].index
df = df[df['matchId'].isin(matches_to_drop)]

In [4]:
first_bans = df[df['order'] <= 6]
first_bans = first_bans.groupby('displayName')['heroName'].count().sort_values(ascending=False)
first_ban_rates = first_bans / len(df['matchId'].unique())

first_picks = df[(df['order'] >= 7) & (df['order'] <= 8)] 
first_picks = first_picks.groupby('displayName')['heroName'].count().sort_values(ascending=False)
first_pick_rates = first_picks / len(df['matchId'].unique())

second_bans = df[(df['order'] >= 9) & (df['order'] <= 11)]
second_bans = second_bans.groupby('displayName')['heroName'].count().sort_values(ascending=False)
second_ban_rates = second_bans / len(df['matchId'].unique())

second_picks = df[(df['order'] >= 12) & (df['order'] <= 17)] 
second_picks = second_picks.groupby('displayName')['heroName'].count().sort_values(ascending=False)
second_pick_rates = second_picks / len(df['matchId'].unique())

third_bans = df[(df['order'] >= 18) & (df['order'] <= 21)]
third_bans = third_bans.groupby('displayName')['heroName'].count().sort_values(ascending=False)
third_ban_rates = third_bans / len(df['matchId'].unique())

third_picks = df[(df['order'] >= 22)] 
third_picks = third_picks.groupby('displayName')['heroName'].count().sort_values(ascending=False)
third_pick_rates = third_picks / len(df['matchId'].unique())

overall_pbs = df.groupby('displayName')['displayName'].count().sort_values(ascending=False)
overal_pbs_rates = overall_pbs / len(df['matchId'].unique())

In [5]:
df[df['isPick'] == True][['isPick', 'order']].value_counts() ## First pick is always 7

isPick  order
True    7        5807
        16       5807
        17       5807
        14       5807
        15       5807
        22       5807
        23       5807
        12       5721
        8        5721
        13       5721
        5          86
        4          86
        6          86
Name: count, dtype: int64

In [6]:
query = '''
    SELECT id, match_details."didRadiantWin"
    FROM match_details
'''
df_match_ids = db.query_select_to_df(query, 'match_details', columns=['match_id', 'didRadiantWin'])

In [7]:
fp_rad_wins = 0 #First pick wins for radiant
fp_rad_total = 0
fp_dire_wins = 0 #Second pick radiant wins
fp_dire_total = 0
for idx, row in df[df['order'] == 7].iterrows():
    if row['isRadiant']:
        if df_match_ids[df_match_ids['match_id'] == row['matchId']]['didRadiantWin'].values[0]:
            fp_rad_wins += 1
        fp_rad_total += 1
    else:
        if not df_match_ids[df_match_ids['match_id'] == row['matchId']]['didRadiantWin'].values[0]:
            fp_dire_wins += 1
        fp_dire_total += 1
fp_rad_winrate = fp_rad_wins / fp_rad_total
fp_dire_winrate = fp_dire_wins / fp_dire_total
fp_total_winrate = (fp_rad_wins + fp_dire_wins) / (fp_rad_total + fp_dire_total)
rad_total_winrate = len(df_match_ids[df_match_ids['didRadiantWin'] == True]) / len(df_match_ids)
dire_total_winrate = len(df_match_ids[df_match_ids['didRadiantWin'] == False]) / len(df_match_ids)
print(f'Radiant first-pick win rate: {fp_rad_winrate}\
    \nDire first-pick win rate: {fp_dire_winrate}\
    \nTotal first-pick win rate: {fp_total_winrate}')
print(f'Radiant win rate overall: {rad_total_winrate}\
      \nDire win rate overall: {dire_total_winrate}')


Radiant first-pick win rate: 0.5490840770314702    
Dire first-pick win rate: 0.4866775421424687    
Total first-pick win rate: 0.5095574306871018
Radiant win rate overall: 0.5264963128108386      
Dire win rate overall: 0.47350368718916136
